# Practical Machine Learning

This module begins with a realistic synthetic churn dataset and builds toward a small model comparison. The students are expected to see the logic of train/test splitting and evaluation before discussing more advanced tuning.


## Build a realistic classification dataset

Each row is a customer. We keep the variables simple, but the relationships are realistic enough to support feature engineering and a real decision workflow.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(17)
customers = pd.DataFrame({
    'months_active': rng.randint(1, 48, size=180),
    'monthly_fee': rng.normal(55, 12, size=180).clip(20, 100),
    'support_calls': rng.poisson(1.5, size=180),
    'contract_month_to_month': rng.binomial(1, 0.55, size=180),
})
logit = -1.2 - 0.045*customers['months_active'] + 0.45*customers['support_calls'] + 0.75*customers['contract_month_to_month']
prob = 1 / (1 + np.exp(-logit))
customers['churned'] = rng.binomial(1, prob)
print(customers.head())
print(round(customers['churned'].mean(), 3))
assert customers.shape == (180, 5)


   months_active  monthly_fee  support_calls  contract_month_to_month  churned
0             16    57.837327              2                        0        0
1              7    56.136020              2                        0        0
2             23    28.931615              3                        1        1
3             46    42.365259              1                        1        0
4             23    68.398007              1                        0        1
0.283


## Train/test split and a baseline

The baseline should be simple and easy to understand. That way, when a more advanced model improves the score, the improvement is easier to interpret.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

X = customers.drop(columns=['churned'])
y = customers['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
base = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(accuracy_score(y_test, base.predict(X_test)))
assert len(X_train) > 0


## Logistic regression and evaluation

A logistic regression model is an excellent teaching model because its output can be explained to students without assuming a black-box framework.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500, random_state=42))
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(confusion_matrix(y_test, preds))
print(round(precision_score(y_test, preds, zero_division=0), 3))
print(round(recall_score(y_test, preds, zero_division=0), 3))
print(round(f1_score(y_test, preds, zero_division=0), 3))
assert len(preds) == len(y_test)


## Cross-validation and practical judgment

This section emphasizes that generalization matters more than fitting the training data perfectly. The model should be tested across folds, not just on one final split.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500, random_state=42)).fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
for name, model in [('logreg', logreg), ('tree', tree)]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    print(name, round(scores.mean(), 3), round(scores.std(), 3))


## Project-style decision memo

Students should end with a short memo that explains the business problem, the model, and the metrics, instead of only coding a pipeline in a vacuum.


In [ ]:
probs = model.predict_proba(X_test)[:, 1]
threshold = 0.35
flagged = (probs >= threshold).astype(int)
print('flagged', int(flagged.sum()))
print('positive share', round(float(flagged.mean()), 3))
assert flagged.shape == (y_test.shape[0],)
